In [68]:
%load_ext autoreload
%autoreload 2


import time
import os
import pandas as pd
import requests
from pprint import pprint

from utils import get_vessel_info, headers, get_vessels_ids, get_port_visitors, parse_port_visits
vesseles = pd.read_csv('../data/shadow_fleet_vessels.csv')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Capacity (DWT ranges)

In [69]:
size_DWT = {
    'Handysize':(7000,40000), #The lower limit has been lowered because we are taking annual ships into account
    'Aframax':(80000,110000),
    'Suezmax':(120000,200000),
}
vesseles['dwt_min'] = vesseles['tanker_size'].map(lambda s:size_DWT[s][0])
vesseles['dwt_max'] = vesseles['tanker_size'].map(lambda s:size_DWT[s][1])
vesseles['dwt_mid'] = (vesseles['dwt_min'] + vesseles['dwt_max']) / 2

vesseles[['vessel_name','tanker_size', 'dwt_min','dwt_max', 'dwt_mid']]


,vessel_name,tanker_size,dwt_min,dwt_max,dwt_mid
0,Aether,Aframax,80000,110000,95000.0
1,Tagor,Aframax,80000,110000,95000.0
2,Phoenix I,Aframax,80000,110000,95000.0
3,Bhilva,Aframax,80000,110000,95000.0
4,Torx,Suezmax,120000,200000,160000.0
5,Aura 1,Aframax,80000,110000,95000.0
6,Balu 6,Aframax,80000,110000,95000.0
7,Jewel,Aframax,80000,110000,95000.0
8,Guanyin,Aframax,80000,110000,95000.0
9,Marjorie,Suezmax,120000,200000,160000.0


In [70]:
#timeless values of voyages range
voyages_range = {
    'Handysize':(8,14),
    'Aframax':(5,8),
    'Suezmax':(4,7),
}
load_factor = 0.95
bbl_per_tone = 7.33

vesseles['voyages_min'] = vesseles['tanker_size'].map(lambda s:voyages_range[s][0])
vesseles['voyages_max'] = vesseles['tanker_size'].map(lambda s:voyages_range[s][1])

vesseles['tonnes_low'] = vesseles['dwt_min']*vesseles['voyages_min']*load_factor
vesseles['tonnes_high'] = vesseles['dwt_max']*vesseles['voyages_max']*load_factor

vesseles['bbl_low'] = vesseles['tonnes_low']*bbl_per_tone
vesseles['bbl_high'] = vesseles['tonnes_high']*bbl_per_tone

In [71]:
urals_price = 66.5

vesseles['value_low'] = vesseles['bbl_low']*urals_price
vesseles['value_high'] = vesseles['bbl_high']*urals_price

by_class = vesseles.groupby('tanker_size')[['value_low','value_high']].sum()

In [72]:
vesseles.head()

,imo,vessel_name,tanker_size,build_year,flag,ship_manager,registered_owner,ism_manager,list_source,dwt_min,dwt_max,dwt_mid,voyages_min,voyages_max,tonnes_low,tonnes_high,bbl_low,bbl_high,value_low,value_high
0,9328170,Aether,Aframax,2007,Unknown,UAE. Zulu Ships Management,UAE. Wrasse Ship Co,UAE. Zulu Ships Management,new_crude_2025,80000,110000,95000.0,5,8,380000.0,836000.0,2785400.0,6127880.0,185229100.0,407504020.0
1,9282481,Tagor,Aframax,2005,Unknown,UAE. Zulu Ships Management,UAE. Blaire Charters Ltd,UAE. Tarabya Logistics Ltd,new_crude_2025,80000,110000,95000.0,5,8,380000.0,836000.0,2785400.0,6127880.0,185229100.0,407504020.0
2,9236248,Phoenix I,Aframax,2002,Comoros,UAE. Triviality Shipping Llc,UAE. Celestial Overseas Co Ltd,UAE. Triviality Shipping Llc,new_crude_2025,80000,110000,95000.0,5,8,380000.0,836000.0,2785400.0,6127880.0,185229100.0,407504020.0
3,9439383,Bhilva,Aframax,2010,Panama,Seychelles. Tika Shipping Ltd,Seychelles. Tika Shipping Ltd,China. Song Hua Jiang Shipmanagement,new_crude_2025,80000,110000,95000.0,5,8,380000.0,836000.0,2785400.0,6127880.0,185229100.0,407504020.0
4,9311610,Torx,Suezmax,2006,Panama,Seychelles. Shiva Shipping Ltd,Seychelles. Shiva Shipping Ltd,Singapore. Crest Maritime Pte Ltd,new_crude_2025,120000,200000,160000.0,4,7,456000.0,1330000.0,3342480.0,9748900.0,222274920.0,648301850.0


## 3. Port visits (GFW)
for ship Tagor


In [73]:

from utils import get_vessel_info, headers

raw_test = get_vessel_info(9282481)
for entry in raw_test['entries']:
    for rec in entry['selfReportedInfo']:
        print(rec['id'],rec['shipname'])

c0830f34f-ff65-a8f1-62c9-1f0e4b069ae5 BRITISH GANNET
e343b1da5-5b7a-de04-e72f-279e95210584 TAGOR
3b359ae29-977a-8747-4fe7-0f0f79269db8 TAGOR
97776be53-3f32-2f42-0439-c5cb44cc945d TAGOR
1ff0ec2ce-ef7a-75f9-7c67-38abeb4f9207 PIPER
a4ab02c0f-f6ed-739d-2413-54991555104c FRENCH NAVY WARSHIP
20c2c5af0-044e-1a07-536f-ebe21ab350b3 PRIAPUS
779363393-3c4a-27ae-e75c-568791c59948 TAGOR


In [74]:
url = "https://gateway.api.globalfishingwatch.org/v3/events"
params = {
    'datasets[0]':'public-global-port-visits-events:latest',
    'vessels[0]': 'e343b1da5-5b7a-de04-e72f-279e95210584',
    'start-date': '2025-01-01',
    'end-date': '2025-12-31',
    'limit': 100,
    'offset': 0
}
r = requests.get(url, headers= headers, params=params )
print (r.text)

{"metadata":{"datasets":["public-global-port-visits-events:v4.0"],"vessels":["e343b1da5-5b7a-de04-e72f-279e95210584"],"dateRange":{"from":"2025-01-01","to":"2025-12-31"}},"limit":100,"offset":0,"nextOffset":null,"total":15,"entries":[{"start":"2024-12-14T05:37:40.000Z","end":"2025-01-03T09:41:13.000Z","id":"80f50b20892a3dcdbebc6cdb37e84c89","type":"port_visit","position":{"lat":59.9233,"lon":28.5965},"regions":{"mpa":[],"eez":["5690"],"rfmo":["NASCO","IWC","ICES","ACAP","ICCAT","NAMMCO"],"fao":["27.3","27","27.3.d","27.3.d.32"],"majorFao":["27"],"eez12Nm":["5690"],"highSeas":[],"mpaNoTakePartial":[],"mpaNoTake":[]},"boundingBox":[28.59650657494041,59.92333973477929,28.600174237032302,59.91691358779298],"distances":{"startDistanceFromShoreKm":8,"endDistanceFromShoreKm":8,"startDistanceFromPortKm":0,"endDistanceFromPortKm":0},"vessel":{"id":"e343b1da5-5b7a-de04-e72f-279e95210584","name":"TAGOR","ssvid":"352002249","flag":"PAN","type":"other","nextPort":null},"port_visit":{"visitId":"3422

In [75]:
data = r.json()
first=data['entries'][0]

first.keys()

dict_keys(['start', 'end', 'id', 'type', 'position', 'regions', 'boundingBox', 'distances', 'vessel', 'port_visit'])

In [76]:


pprint(first['port_visit'])

{'confidence': '4',
 'durationHrs': 484.05916666666667,
 'endAnchorage': {'anchorageId': '46969735',
                  'atDock': False,
                  'distanceFromShoreKm': '8',
                  'flag': 'RUS',
                  'id': 'rus-ustluga',
                  'lat': 59.91691358779298,
                  'lon': 28.600174237032302,
                  'name': None,
                  'topDestination': 'UST LUGA'},
 'intermediateAnchorage': {'anchorageId': '46969731',
                           'atDock': False,
                           'distanceFromShoreKm': '8',
                           'flag': 'RUS',
                           'id': 'rus-ustluga',
                           'lat': 59.92333973477929,
                           'lon': 28.59650657494041,
                           'name': None,
                           'topDestination': 'UST LUGA'},
 'startAnchorage': {'anchorageId': '46969731',
                    'atDock': False,
                    'distanceFromShoreKm': '

In [77]:
rows = []
for e in data ['entries']:
    anch = e['port_visit']['intermediateAnchorage']
    rows.append({
        'start': e['start'],
        'port': anch['topDestination'],
        'country': anch['flag'],
        'hours':e['port_visit']['durationHrs'],
        'at_dock': anch['atDock']
    })

visits = pd.DataFrame(rows)
visits

# Rough tanker port-stay heuristics (hours):
# <6    short stop (crew, supplies, pilot)
# 6-24  canal transit / bunkering
# 24-48 loading or discharge at berth
# 48-120 discharge + queue / STS transfer
# 120-360 waiting for orders / floating storage
# >360  repair / drydock / long idle

,start,port,country,hours,at_dock
0,2024-12-14T05:37:40.000Z,UST LUGA,RUS,484.059167,False
1,2025-01-04T21:50:35.000Z,PRIMORSK,RUS,28.352500,False
2,2025-01-06T05:14:57.000Z,PRIMORSK,RUS,36.007222,False
3,2025-01-11T23:48:06.000Z,SKAGEN,DNK,8.194167,False
4,2025-02-03T10:37:37.000Z,PORT SAID,EGY,17.010556,False
5,2025-02-15T16:15:21.000Z,MUNDRA,IND,114.372500,False
6,2025-02-25T09:30:25.000Z,FUJAIRAH,ARE,202.173333,False
7,2025-03-08T11:07:23.000Z,AL DUQUM OMAN,OMN,709.354167,False
8,2025-04-15T22:58:43.000Z,SUEZ,EGY,31.729444,False
9,2025-04-30T14:52:31.000Z,SKAGEN,DNK,15.313056,False


### Can `atDock` identify loadings?
No. `atDock` is `False` for all RUS port visits of Tagor, including Russian oil terminals
where the vessel must have loaded. The flag is not reliable in GFW data, so it is not used.

**How loadings are identified instead:**
- port country = `RUS`
- stay of roughly 24–48 hours (typical loading time for an Aframax)
- consecutive visits to the same port are counted as one loading (e.g. Primorsk, 4 and 6 Jan)

In [78]:
ru = visits[visits['country'] == 'RUS'].drop(columns=['at_dock'])
loadings = ru[(ru['hours'] >= 12) & (ru['hours'] <= 120)]
loadings = loadings[loadings['port']!=loadings['port'].shift()]
loadings

,start,port,country,hours
1,2025-01-04T21:50:35.000Z,PRIMORSK,RUS,28.352500
10,2025-05-15T09:55:00.000Z,UST LUGA,RUS,48.891111
14,2025-07-27T15:25:41.000Z,KOZMINO,RUS,40.325833


Tagor: **3 loadings** in 2025 (Primorsk ×2 visits = 1 loading).


In [79]:
len(loadings)

3

In [80]:
n_loadings = len(loadings)
tagor = vesseles[vesseles['vessel_name']=='Tagor'].iloc[0]

real_low = n_loadings*tagor['dwt_min']*load_factor*bbl_per_tone*urals_price
real_high =n_loadings*tagor['dwt_max']*load_factor*bbl_per_tone*urals_price

print(real_low/1e6, real_high/1e6)

111.13746 152.8140075


### Tagor: real loadings vs assumption
| | Loadings | Cargo value |
|---|---|---|
| Assumption (section 2) | 5–8 / year | $185–408M |
| GFW port visits | 3 in ~7 months (≈5 / year) | **$111–153M** |

The assumption was reasonable, but on the high side.

This is a **lower bound**:
- data ends on 27 Jul (AIS could be off after that)
- only 1 of Tagor's 8 GFW ids was used
- the 20-day Ust-Luga stay was not counted (possible STS)

In [81]:
#chek def get_port_visitors and parse_port_visitors for ship TAGOR
get_vessels_ids(9282481)

['c0830f34f-ff65-a8f1-62c9-1f0e4b069ae5',
 'e343b1da5-5b7a-de04-e72f-279e95210584',
 '3b359ae29-977a-8747-4fe7-0f0f79269db8',
 '97776be53-3f32-2f42-0439-c5cb44cc945d',
 '1ff0ec2ce-ef7a-75f9-7c67-38abeb4f9207',
 'a4ab02c0f-f6ed-739d-2413-54991555104c',
 '20c2c5af0-044e-1a07-536f-ebe21ab350b3',
 '779363393-3c4a-27ae-e75c-568791c59948']

In [82]:
ids = get_vessels_ids(9282481)
entries = get_port_visitors(ids)
len(entries)

16

In [83]:
rows = parse_port_visits(9282481, entries)
tagor_all = pd.DataFrame(rows).sort_values('start')
tagor_all

,imo,start,port,country,hours
0,9282481,2024-12-14T05:37:40.000Z,UST LUGA,RUS,484.059167
1,9282481,2025-01-04T21:50:35.000Z,PRIMORSK,RUS,28.352500
2,9282481,2025-01-06T05:14:57.000Z,PRIMORSK,RUS,36.007222
3,9282481,2025-01-11T23:48:06.000Z,SKAGEN,DNK,8.194167
4,9282481,2025-02-03T10:37:37.000Z,PORT SAID,EGY,17.010556
5,9282481,2025-02-15T16:15:21.000Z,MUNDRA,IND,114.372500
6,9282481,2025-02-25T09:30:25.000Z,FUJAIRAH,ARE,202.173333
7,9282481,2025-03-08T11:07:23.000Z,AL DUQUM OMAN,OMN,709.354167
8,9282481,2025-04-15T22:58:43.000Z,SUEZ,EGY,31.729444
9,9282481,2025-04-30T14:52:31.000Z,SKAGEN,DNK,15.313056


In [84]:

PATH = '../data/port-visits.csv'
if os.path.exists(PATH):
    port_visits = pd.read_csv(PATH)
else:
    all_rows = []
    for imo in vesseles['imo']:
        ids = get_vessels_ids(imo)
        entries = get_port_visitors(ids)
        all_rows.extend(parse_port_visits(imo, entries))
        time.sleep(0.5)

    port_visits = pd.DataFrame(all_rows)
    port_visits.to_csv(PATH, index=False)

port_visits.shape


(1755, 5)

In [85]:
#how many ships were in russia port
ru_all = port_visits[port_visits['country']=='RUS']
loadings_all = ru_all[(ru_all['hours']>=12)&(ru_all['hours']<=120)]
loadings_all = loadings_all.sort_values(['imo', 'start'])
loadings_all = loadings_all[loadings_all['port']!=loadings_all.groupby('imo')['port'].shift()]
loadings_all


,imo,start,port,country,hours
915,9037123,2025-05-19T11:53:27.000Z,KOZMINO,RUS,76.063056
925,9037123,2025-08-22T21:04:55.000Z,VANINO,RUS,79.505556
930,9037123,2025-10-25T22:50:47.000Z,RU VNN,RUS,53.876389
1246,9046564,2025-01-03T06:02:43.000Z,OKHOTSK SEA,RUS,80.398056
1251,9046564,2025-02-23T00:20:05.000Z,SLAVYANKA,RUS,38.223611
...,...,...,...,...,...
1622,9884150,2025-09-01T17:17:16.000Z,YEISK,RUS,54.613889
1627,9884150,2025-09-21T11:49:57.000Z,KAVKAZ,RUS,18.052500
1631,9884150,2025-10-09T11:02:59.000Z,KERCH,RUS,22.727222
1637,9884150,2025-11-07T07:49:02.000Z,KAVKAZ,RUS,43.660556


In [91]:
n_load = loadings_all.groupby('imo').size().rename('n_loadings')
money = vesseles.merge(n_load, on='imo', how='left')
# River-sea shuttles: each round trip = 2 Russian port visits → count half
is_handy = money['tanker_size'] == "Handysize"
money.loc[is_handy, 'n_loadings'] = money.loc[is_handy, 'n_loadings'] / 2
money['n_loadings'] = money['n_loadings'].fillna(0)

money[['vessel_name', 'tanker_size', 'n_loadings']].sort_values('n_loadings', ascending=False)


,vessel_name,tanker_size,n_loadings
35,Thoth,Handysize,25.5
33,Vasily Surikov,Handysize,16.5
47,Sergey Lvov,Handysize,10.0
46,Bm Flot-3,Handysize,9.0
37,Antares,Handysize,9.0
36,Midvolga-3,Handysize,7.0
14,Centurion I,Aframax,7.0
34,Professor Malyshkin,Handysize,6.5
48,Viktor Vasnetsov,Handysize,5.5
49,Pavel Bryullov,Handysize,5.5


Handysize tankers (mostly river-sea) shuttle between two Russian ports
(e.g. Thoth: St Petersburg ↔ Ust-Luga), so each round trip shows up as 2 visits.
Their loadings are divided by 2.


In [94]:
money['real_low'] = money['n_loadings']*money['dwt_min']*load_factor*bbl_per_tone*urals_price
money['real_high'] = money['n_loadings']*money['dwt_max']*load_factor*bbl_per_tone*urals_price

print(money[['real_low', 'real_high']].sum() / 1e9)
money.groupby('tanker_size')[['n_loadings','real_low','real_high']].sum()

by_class_real = money.groupby('tanker_size')[['n_loadings', 'real_low', 'real_high']].sum()
by_class_real[['real_low', 'real_high']] = (by_class_real[['real_low', 'real_high']] / 1e9).round(2)
by_class_real
#real_low and realy_high show in bln usd

real_low     4.028964
real_high    7.497148
dtype: float64


,n_loadings,real_low,real_high
tanker_size,,,
Aframax,63.0,2.33,3.21
Handysize,111.5,0.36,2.07
Suezmax,24.0,1.33,2.22


In [98]:
(money[['real_low', 'real_high']].sum() / 1e9).round(2)

real_low     4.03
real_high    7.50
dtype: float64